# Inference with the PPAN Model
This notebook goes through the steps required to perform inference with 
the PPAN Model.
The process involves a few steps:

1. Scanning a directory for all videos
2. Iteratively Loading and Pre-processing all videos
3. Getting predictions from the model
4. Performing post-processing on the model predictions
5. Generating a nice MIDI file

Luckily, the PPAN python package contains many helper functions we can 
utilize!

In [ ]:
# Let's get imports out of the way first
import os
import pickle
from pathlib import Path
from typing import Optional

import numpy.typing as npt
from torch import no_grad
from transformers import (
    VideoMAEImageProcessor
)

from ppan.config import pretrained_model
from ppan.dataset import PPAnEvalDataset
from ppan.evaluate_model import (
    eval_loop, 
    calc_time, 
    final_pred_to_onset_array,
    save_to_midi
)

# Useful for typing
PathLike = str | os.PathLike

## Parameters
Make sure that all videos you are using are the same framerate and that 
the framerate is a multiple of 30. 
They should also all be in a format supported by the Nvidia DALI library.


In [ ]:
# Dataset settings
## Folder containing all videos.
video_dir: PathLike = None
## Whether to rotate the video 180 degrees.
rotate_180: Optional[bool] = None  

# Output settings
## Directory to output generated MIDI files to. 
midi_output_dir: Optional[PathLike] = None
## Pickle file to store model preds.
cache_output_file: Optional[PathLike] = None
## Whether to recalculate the model preds even if the cache file exists.
recalc_preds: Optional[bool] = None

# Model settings
model_checkpoint: PathLike = None
batch_size: int = None

# Settings for model output post-processing
threshold: Optional[float] = None
gaussian_sigma: Optional[float] = None

In [ ]:
# Check everything is set correctly and do some initial setup.
required_vars = [video_dir, recalc_preds, model_checkpoint, batch_size]
if None in required_vars:
    raise AttributeError("Please set all required variables!!")

# Set defaults:
if cache_output_file is None:
    cache_output_file = "model_preds.pkl"
if midi_output_dir is None:
    midi_output_dir = "./preds_midi"
if recalc_preds is None:
    recalc_preds = False
if threshold is None:
    threshold = 0.5
if gaussian_sigma is None:
    gaussian_sigma = 1
    
# If the MIDI output directory doesn't exist we create it:
midi_output_dir = Path(midi_output_dir)
midi_output_dir.mkdir(exist_ok=True)

## Data Loading
Let's write a helper function for loading our data.
Because we'd like to use the PPAnEvalDataset object, we need to make sure 
our data is in the correct format.

In [ ]:
def prepare_data(data_dir: PathLike, rotate: bool = False):
    """
    Simple sample loading script. Feel free to modify this for your 
    needs.
    
    Parameters
    ----------
    data_dir : PathLike
        Folder containing videos in a format supported by Nvidia DALI.
    rotate : bool, optional

    Returns
    -------
    samples : ppan.dataset.SAMPLE_TYPE
        [midi_path, flac_path, video_path, bounding_box, whether to rotate 180, random_resize]
    """
    videos = os.listdir(data_dir)
    none = [None for _ in videos]
    rotate = [rotate for _ in videos]
    false = [False for _ in videos]
    return list(zip(none, none, videos, none, rotate, false))

Now let's use our helper function and create an instance of the dataset.

In [ ]:
# Load all our samples
samples = prepare_data(data_dir=video_dir, rotate=rotate_180)

# Load our dataset and pre-trained model processor
processor = VideoMAEImageProcessor.from_pretrained(
    pretrained_model
)
dataset = PPAnEvalDataset(
    datasets=samples,
    video_transform=processor,
    batch_size=batch_size
)

## Inference
With the dataset loaded, we can now perform inference using the 
trained model.
For this we utilize the provided helper function from PPAN.

In [ ]:
if not Path(cache_output_file).exists() or recalc_preds:
    # Use no_grad for better performance.
    with no_grad():
        preds = eval_loop(
            dataset=dataset,
            model_checkpoint=model_checkpoint
        )
    # Save our outputs so that we can rerun this notebook with the 
    # cached outputs.
    with open(cache_output_file, "wb") as f:
        pickle.dump(obj=preds, file=f)
else:
    # Load our already calculated model predictions.
    with open(cache_output_file, "rb") as f:
        preds = pickle.load(f)

## Post-Processing
Although we have model predictions, there's still work to do to make 
them usable.
Let's define a helper function for post-processing the model outputs.

In [ ]:
def do_postprocessing(video_preds, threshold, gaussian_sigma):
    """
    Perform post-processing on the PPAN model outputs.
    This function utilizes the provided functions from PPAN.
    
    Returns
    -------
    onset_array : npt.NDArray[int]
        A regular onset array, similar to a pianoroll.
    """
    final_pred = calc_time(video_preds)
    onset_array = final_pred_to_onset_array(
        final_pred, 
        threshold,
        gaussian_sigma
    )
    # Multiply the onset array by 100 to make the notes louder.
    onset_array = onset_array.astype(int) * 100
    return onset_array

## Saving to MIDI
Let's also define a helper function that saves our predictions to a MIDI 
file.

In [ ]:
def save_to_midi(onset_array: npt.NDArray[int], 
                 video_path: PathLike, 
                 midi_output_dir: Path) -> None:
    """
    Save an onset_array to a MIDI file. 
    Utilizes a helper function from PPAN.
    
    Parameters
    ----------
    video_path : PathLike
    midi_output_dir : Path
    """
    # Let's name the MIDI file after the input video.
    vid_path = Path(video_path)
    mid_output = midi_output_dir/(vid_path.stem + ".mid")
    save_to_midi(onset_array, str(mid_output))

## Performing Post-Processing and Saving MIDI's
With our helper functions we can iterate through our model predictions 
and apply post-processing as well as save them. 

In [ ]:
# Iterate through all videos and their predictions.
for vid_path, vid_preds in preds.items():
    onset_array = do_postprocessing(
        video_preds=vid_preds,
        threshold=threshold,
        gaussian_sigma=gaussian_sigma
    )
    save_to_midi(
        onset_array=onset_array,
        video_path=vid_path,
        midi_output_dir=midi_output_dir
    )

## Conclusion
With that we're done! 
You can listen to your shiny new MIDI files in any way you'd like.
One can see that PPAN's helper functions make inference much easier.
The only thing that the may need to change is the way samples are 
loaded.